# Exercises — Chapter 12: Explainability and XAI in Finance

Complete the exercises from Lecture 12.

In [ ]:
# Your code here

## Data Lab — SEC EDGAR

Build an explainable classifier on EDGAR fundamentals and visualise feature attributions.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

LM_POS = {"profitable","strong","growth","improved","exceeded","innovative",
           "efficient","leading","record","gained","successful","robust",
           "increased","advancing","outperformed","benefit","enhanced","positive","achieved","favourable"}
LM_NEG = {"risk","loss","uncertain","decline","adverse","default","impair",
           "volatile","litigation","breach","failed","reduced","weakness",
           "debt","distress","penalty","violation","exposure","downturn","writedown"}


### Exercise [B]: Logistic Regression Coefficients

In [ ]:
# --- Exercise [B]: Logistic Regression on EDGAR Ratios ---
TICKERS_12 = ["AAPL","MSFT","GOOGL","JPM","BAC","XOM","JNJ","PFE","GE","F"]

def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-20,20)))

feat_rows, label_rows = [], []
for t in TICKERS_12:
    try:
        cik = get_cik(t)
        def gv(c, fb=0):
            try: return get_annual_series(cik,c).sort_values("date").iloc[-1][c]
            except: return fb
        A  = gv("Assets")    or 1
        E  = gv("StockholdersEquity") or 1
        NI = gv("NetIncomeLoss")
        D  = gv("LongTermDebt")
        try:
            rev_s = get_annual_series(cik,"Revenues").sort_values("date")["Revenues"]
            rg = float(rev_s.iloc[-1]/rev_s.iloc[-2]-1) if len(rev_s)>=2 else 0
        except: rg=0
        leverage = D/A; roe = NI/abs(E) if E!=0 else 0
        feat_rows.append([leverage, roe, rg])
        # Approximate label from public knowledge
        label_rows.append(1 if t in ["AAPL","MSFT","GOOGL","JPM","JNJ"] else 0)
    except Exception as e:
        print(f"{t}: {e}"); feat_rows.append([0,0,0]); label_rows.append(0)

X12 = np.array(feat_rows); y12 = np.array(label_rows, dtype=float)
X12 = (X12 - X12.mean(0)) / (X12.std(0)+1e-8)
Xb  = np.c_[np.ones(len(X12)), X12]
w12 = np.zeros(Xb.shape[1])
for _ in range(1000):
    p = sigmoid(Xb @ w12); w12 -= 0.3 * Xb.T@(p-y12)/len(y12)

print("Coefficients [bias, leverage, ROE, rev_growth]:")
for name, coef in zip(["bias","leverage","ROE","rev_growth"], w12):
    print(f"  {name:<15} {coef:+.4f}")

### Exercise [I]: SHAP-Style Feature Attribution

In [ ]:
# --- Exercise [I]: Leave-One-Feature-Out Attribution ---
means = X12.mean(0)
feat_names = ["leverage","ROE","rev_growth"]

def pred_prob(x_row):
    return sigmoid(np.r_[1, x_row] @ w12)

attributions = []
for i, t in enumerate(TICKERS_12[:3]):
    x   = X12[i]
    p_full = pred_prob(x)
    attrs  = {}
    for j, name in enumerate(feat_names):
        x_drop     = x.copy(); x_drop[j] = means[j]
        p_drop     = pred_prob(x_drop)
        attrs[name] = p_full - p_drop
    attributions.append({"ticker":t, "p_full":p_full, **attrs})
    print(f"{t}: P={p_full:.3f}  {attrs}")

# Waterfall for first company
t0 = attributions[0]
fig, ax = plt.subplots(figsize=(7,4))
vals = [t0[f] for f in feat_names]
colors = ["green" if v>0 else "tomato" for v in vals]
ax.bar(feat_names, vals, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Attribution (Delta P)"); ax.set_title(f"Feature Attribution — {attributions[0]['ticker']}")
plt.tight_layout(); plt.show()

### Exercise [A]: Model Card

In [ ]:
# --- Exercise [A]: Model Card ---
loo_correct = 0
for i in range(len(X12)):
    mask  = np.arange(len(X12))!=i
    Xtr   = np.c_[np.ones(mask.sum()), X12[mask]]; ytr=y12[mask]
    w     = np.zeros(Xtr.shape[1])
    for _ in range(500):
        p=sigmoid(Xtr@w); w-=0.3*Xtr.T@(p-ytr)/len(ytr)
    loo_correct += int(int(sigmoid(np.r_[1,X12[i]]@w)>0.5)==int(y12[i]))

model_card = {
    "model_type":         "Logistic Regression (NumPy, gradient descent)",
    "training_data":      "EDGAR XBRL annual 10-K filings, 10 S&P 500 companies, latest fiscal year",
    "features": [
        {"name":"leverage",    "xbrl":"LongTermDebt / Assets"},
        {"name":"ROE",         "xbrl":"NetIncomeLoss / StockholdersEquity"},
        {"name":"rev_growth",  "xbrl":"YoY change in Revenues"},
    ],
    "performance":        f"LOO-CV accuracy: {loo_correct}/{len(X12)} = {loo_correct/len(X12):.2f}",
    "known_limitations":  [
        "n=10 — far too small for production deployment",
        "Labels are approximated from public rating knowledge, not validated ratings",
        "Single fiscal year: no temporal cross-validation",
        "Survivorship bias: only S&P 500 companies (no defaults observed)",
    ],
    "intended_use":       "Educational illustration of explainability techniques on financial data",
    "out_of_scope_use":   "Actual credit decisions, investment recommendations, or regulatory filings",
}

import json
print(json.dumps(model_card, indent=2))